# More Exploring of Data To Answer the Hypothesis

## Hypothesis

Using only the Epstein files, what connections can I make between people who have an appearance in the files.

In [1]:
import duckdb
import pandas as pd
import glob
import json
from collections import defaultdict

In [83]:
## DOCS -> a dictionary of {document_type: parquet_path}
DOCS = {
    "court" : "../data/court_documents.parquet",
    "emails": "../data/email_documents.parquet",
    "logs": "../data/log_documents.parquet",
    "myspace": "../data/myspace_documents.parquet",
    "reports": "../report_documents.parquet"
}

In [101]:
## Can I map map names to a normalized version?
query = f"""
    WITH
        individuals AS (
            SELECT
                unnest(people) AS person
            FROM
                '{DOCS["emails"]}'
        )
    SELECT person FROM individuals ORDER BY random()
"""

people = duckdb.query(query).df()

In [88]:
target = "Lofthus"

query = f"""
    SELECT
        person
    FROM
        people
    WHERE
        person ILIKE '%{target}%'
"""

duckdb.query(query)

┌─────────────┐
│   person    │
│   varchar   │
├─────────────┤
│ Lee Lofthus │
└─────────────┘

In [ ]:
# Manually mapping aliases (Only some)
NAME_MAP = {
    "Martin Licon-Vitale": ["Marti Licon-Vitale", "Licon-Vitale", "Marti Licom-Vitale"],
    "Jeffrey Epstein": ["Epstein", "Inmate Epstein, J.", "Jeffrey EPSTEIN", "JEFFREY EDWARD EPSTEIN", "EPSTEIN, Jeffrey", "Epstein, Jeffrey Edward", "Mr. Epstein", "Mr. E", "Jeffrey Edward Epstein", "EPSTEIN",
                        "Jeffrey Epstei", "Jeffrey E.", "Jeffrey Edward Epstein"],
    "Darren Indyke" : ["Darren Indyk"],
    "Bernard Madoff" : ["Bernie Madoff"],
    "Benedict H. Gross" : ["Dr. Benedict H. Gross"],
    "Jeffrey Keller": ["Mr. Keller", "Jeffery Keller"],
    "Jonathan David Farley" : ["Dr. Jonathan David Farley"],
    "Bobbi C Sternheim" : ["Bobbi"],
    "Hugh Hurwitz": ["Hugh Hurwit", "Mr. Hurwitz", "Hurwitz Hugh"],
    "William Mook": ["Bill Mook"],
    "James Wills": ["James C. Wills", "James Will"],
    "Lee Plourde": ["Plourde Lee"],
    "Judge Sullivan": ["J. Sullivan"],
    "Ray Ormond": ["J. Ray Ormond", "Mr. Ormond", "Ormond Ray"],
    "Dr. S. Allen Counter": ["S. Allen Counter"],
}

REVERSED_NAME_MAP = {x: k for k,v in NAME_MAP.items() for x in v}

In [ ]:
def mapNames(name: str):
    if (not isinstance(name, str)):
        return None
    
    name = name.strip()
    
    try:
        return REVERSED_NAME_MAP[name]
    except KeyError:
        return name
    

# people["person"] = people["person"].apply(mapNames)

TypeError: list indices must be integers or slices, not str

## Who Was Epstein In Frequent Communication With?

In [ ]:
query = f"""
    SELECT
        person,
        COUNT(*) AS freq
    FROM
        people
    WHERE
        person != 'Jeffrey Epstein'
    GROUP BY
        person
    ORDER BY
        freq DESC
"""

duckdb.query(query).df().to_csv("./frequent_communication.csv")

# This is kinda underwhelming. The top people are court related?

In [7]:
all_files = glob.glob("../docs/**/*.json")

emails = set()

for file_name in all_files:
    with open(file_name, "rt", encoding="utf-8") as f:
        data = json.load(f)
    
    metadata = data["document_metadata"]

    dt = metadata["document_type"].lower() if metadata["document_type"] is not None else None

    if (dt and dt == "email"):
        emails.add(file_name)

emails = list(emails)

print(len(emails))
print(emails[0])

1066
../docs/IMAGES010/DOJ-OGR-00026759.json


In [17]:
# I want to map people to each other

PEOPLE_MAP = defaultdict(lambda: defaultdict(int))

for file_name in emails:
    with open(file_name, "rt", encoding="utf-8") as f:
        data = json.load(f)

    try:
        entities = data["entities"]

        people = entities["people"]

        for control in people:
            control = mapNames(control)
            for target in people:
                target = mapNames(target)
                if (control == target):
                    continue

                PEOPLE_MAP[control][target] += 1
    except Exception as e:
        print(e)

name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'mapNames' is not defined
name 'ma

In [16]:
PEOPLE_MAP

defaultdict(<function __main__.<lambda>()>,
            {'Shirley V. Skipper-Scott': defaultdict(int,
                         {'Jeffrey Epstein': 53,
                          "Lamine N'Diaye": 116,
                          'Lee Plourde': 45,
                          'Charisma Edge': 76,
                          'Ray Ormond': 83,
                          'J. Ray Ormond': 37,
                          'Epstein': 116,
                          'James Petrucci': 26,
                          'Mike': 7,
                          'Jeffrey Edward Epstein': 17,
                          'S. Allen Counter': 1,
                          'Benedict H. Gross': 1,
                          'Jonathan David Farley': 3,
                          "Lamnie N'Diaye": 1,
                          'Jeffrey EPSTEIN': 2,
                          'Marti Licon-Vitale': 11,
                          'Rocco Lupo': 1,
                          'Roberto Maisonet': 1,
                          'Rosa Proto': 2,